# Backtracking Solver

This notebook demonstrates the `BacktrackingSolver` — a recursive solver with the **Minimum Remaining Values (MRV)** heuristic.

## How MRV works

At each step, instead of picking the first empty cell we find, we pick the empty cell with the **fewest valid candidates**. This is the MRV (also called *fail-first*) heuristic:

- Cells with fewer options are more constrained — they'll cause a contradiction sooner if we chose wrong.
- Detecting contradictions early prunes large branches of the search tree.
- In practice, MRV reduces the effective search space dramatically compared to naive left-to-right scanning.

For Sudoku, a candidate is a digit 1–9 not already present in the cell's row, column, or 3×3 box.

In [ ]:
%pip install ipympl -q

In [ ]:
import time
import pandas as pd
from sudoku import BacktrackingSolver, Sudoku
from sudoku.game.presets import EASY, HARD, MEDIUM

solver = BacktrackingSolver()

## Solve all three presets

In [ ]:
for label, puzzle_str in [("EASY", EASY), ("MEDIUM", MEDIUM), ("HARD", HARD)]:
    puzzle = Sudoku.from_string(puzzle_str)
    t0 = time.perf_counter()
    solution = solver.solve(puzzle)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    print(f"\n=== {label} — solved in {elapsed_ms:.2f} ms ===")
    print("Puzzle:")
    puzzle.display()
    print("Solution:")
    solution.display()

## Benchmark

Run each preset 10 times and collect aggregate timing statistics.

In [ ]:
N_RUNS = 10
rows = []
for label, puzzle_str in [("easy", EASY), ("medium", MEDIUM), ("hard", HARD)]:
    puzzle = Sudoku.from_string(puzzle_str)
    stats = solver.benchmark([puzzle], n_runs=N_RUNS)
    rows.append({
        "preset": label,
        "solve_rate": stats["solve_rate"],
        "mean_ms": stats["mean_time_s"] * 1000,
        "min_ms":  stats["min_time_s"]  * 1000,
        "max_ms":  stats["max_time_s"]  * 1000,
        "n_runs":  stats["n_runs"],
    })

df = pd.DataFrame(rows).set_index("preset")
df.style.format({
    "solve_rate": "{:.0%}",
    "mean_ms": "{:.2f}",
    "min_ms": "{:.2f}",
    "max_ms": "{:.2f}",
})